In [ ]:
import sys
from pathlib import Path
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root))

from src.downloadData import download_dataset_csvs
summary = download_dataset_csvs(
    output_dir=repo_root /"data" /"raw",
    progress=True,
)

print(f"downloaded={len(summary.downloaded_paths)}, skipped={len(summary.skipped_paths)}")

In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType

import os
import sys
import shutil

python_exe = sys.executable
os.environ["PYSPARK_PYTHON"] = python_exe
os.environ["PYSPARK_DRIVER_PYTHON"] = python_exe

spark = (
    SparkSession.builder
    .appName("EEG_Schizoprenia")
    .master("local[*]")
    .config("spark.pyspark.python", python_exe)
    .config("spark.pyspark.driver.python", python_exe)
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.memory", "6g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.files.maxPartitionBytes", "64m")
    .getOrCreate()
)

sc = spark.sparkContext
print("Spark started:", spark.version)
print("Python:", sc.pythonExec)

In [ ]:
base_path = ".small/raw"
data_paths = list(Path(base_path).rglob("*/*.csv"))
print(f"Found {len(data_paths)} files. " \
      f"From `{data_paths[0]}` to `{data_paths[-1]}`")

In [ ]:
column_names = spark.read.csv(r".\data\raw\columnLabels.csv",
                              header=True).columns
print("List of column labels:")
for i, column in enumerate(column_names, start=1):
    print(f"  {i}. {column}" )

In [ ]:
df = spark.read.csv(r".small\raw\*\*.csv", 
                    inferSchema=True, # infer data types
                    header=False)
df = df.toDF(*column_names) 

row_count = df.count()
print(f"Read {row_count:,} records")

df.show(5)

In [ ]:
# from pyspark.sql.functions import avg, max, min, stddev

# group_cols = ["subject", "trial", "condition"]
# other_cols = [col_name for col_name in df.columns if col_name not in group_cols]
# other_cols.remove("sample")

# grp_functions = [(avg, "avg"),(max, "max"),(min, "min"),(stddev, "stddev")]

# agg_df = df.groupBy(group_cols).agg(
#                 *[f(c).alias(f"{c}_{name}") for c in other_cols for f, name in grp_functions]
#             ).orderBy(group_cols)
# agg_df.show(15)

# # In here I wanted to add mode as well but since mode is a heavy process, it caused memory errors
# # I also tried adding q1, median, and q3 but it was also too heavy for our local machines

In [ ]:
from pyspark.ml.feature import StandardScaler, VectorAssembler
from pyspark.ml.functions import vector_to_array
from pyspark.sql.functions import col

no_normalize = ["subject", "trial", "condition", "sample"]
normalize_cols = [col_name for col_name in df.columns if col_name not in no_normalize]

# Step 1: Assemble
assembler = VectorAssembler(inputCols=normalize_cols, outputCol="features")
df_vec = assembler.transform(df)

# Step 2: Scale
scaler = StandardScaler(
    inputCol="features",
    outputCol="scaledFeatures",
    withStd=True,
    withMean=True
)
scalerModel = scaler.fit(df_vec)
df_vec = scalerModel.transform(df_vec)

# Step 3: Convert vector → array
df_vec = df_vec.withColumn("scaled_array", vector_to_array(col("scaledFeatures")))

# Step 4: Extract back to columns
for i, c in enumerate(normalize_cols):
    df_vec = df_vec.withColumn(c, col("scaled_array")[i])

# Step 5: Final dataframe
normalized_df = df_vec.select(no_normalize + normalize_cols)
normalized_df.show(5)

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number, count, floor, avg, when

# Window for ordering
window_spec = Window.partitionBy("subject", "trial", "condition").orderBy("sample")
df_grouped = normalized_df.withColumn("row_num", row_number().over(window_spec))

# Total count per group
count_window = Window.partitionBy("subject", "trial", "condition")

df_grouped = df_grouped.withColumn(
    "total_count",
    count("*").over(count_window)
)

# Compute group size
df_grouped = df_grouped.withColumn(
    "group_size",
    col("total_count") / 30
)

# Assign group id
df_grouped = df_grouped.withColumn(
    "group_id",
    floor((col("row_num") - 1) / col("group_size"))
)

# Fix overflow
df_grouped = df_grouped.withColumn(
    "group_id",
    when(col("group_id") >= 30, 29).otherwise(col("group_id"))
)

# Aggregate
agg_exprs = [avg(c).alias(c) for c in normalize_cols]

paa_df = df_grouped.groupBy(
    "subject", "trial", "condition", "group_id"
).agg(*agg_exprs)

paa_df.show()

In [ ]:
def discretize(df, cols):
    from pyspark.sql.functions import when, col

    for c in cols:
        df = df.withColumn(
            f"{c}_sax",
            when(col(c) < -0.67, "a")
            .when((col(c) >= -0.67) & (col(c) < 0), "b")
            .when((col(c) >= 0) & (col(c) < 0.67), "c")
            .otherwise("d")
        )
    return df

binned_df = discretize(paa_df, normalize_cols)
binned_df.show(5)

In [ ]:
import pyspark.sql.functions as f

# Columns
group_cols = ["subject", "trial", "condition"]
order_col = "group_id"

# Numeric columns (PAA source)
# assuming you still have numeric version before binning OR stored separately
# if not, replace with your numeric df (important!)
paa_cols = normalize_cols  

# SAX columns (same names after binning)
sax_cols = [f"{c}_sax" for c in normalize_cols]


# Helper: ordered collect_list
def ordered_collect(col_name):
    return f.expr(f"""
        transform(
            sort_array(collect_list(struct({order_col}, {col_name}))),
            x -> x.{col_name}
        )
    """)


# Build aggregations
agg_exprs = []

for c in sax_cols:
    # SAX (string of letters)
    agg_exprs.append(
        f.concat_ws(
            "",
            ordered_collect(c)
        ).alias(c)
    )


# Final aggregation
sax_df = binned_df.groupBy(group_cols).agg(*agg_exprs)

sax_df.show(5)

In [ ]:
sax_df.coalesce(1).write.mode("overwrite").option("header", True).csv("data/processed/1-4_csv_sax")

In [ ]:
spark.stop()